In [ ]:
# ============================================================
# Cell 1 — Imports
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

import glob, os
import numpy as np
import pandas as pd

In [ ]:
# ============================================================
# Cell 2 — Configuration  (edit this cell to match your setup)
# ============================================================

# --- File paths ---
# Adjust filenames/paths to match your actual .dat files
# FILES = {
#     'Sample 452': {
#         'pre':  '452_unannealed.DAT',
#         'post': '452_annealed.DAT',
#     },
#     'Sample 270': {
#         'pre':  '270_unannealed.DAT',
#         'post': '270_annealed.DAT',
#     },
# }

FILES = {
    '[Co(4)/Ni(5)/Pt(2)]': {
        # 'Out of plane':  '452_unannealed.DAT',
        'Out of plane':  '452_OOP_20000_11NOV26.DAT',
        'In plane': 'CO4NI5PT2_IP_20000_7OCT26.DAT',
    },
    '[Co(2)/Ni(7)]': {
        'Out of plane':  '270_OOP_20000_11NOV26.DAT',
        'In plane': '270_IP_STRIPE_24JUNE26.DAT',
    },
    '[Co(6)/Ni(6)/Pt(5)]': {
        'Out of plane':  '665_OOP_20000_11NOV26.DAT',
        'In plane': 'CO6NI6PT5_IP_20000_7OCT26.DAT',
    },
    '[Co(10)/Pt(5)]': {
        'Out of plane':  '105_OOP_20000_11NOV26.DAT',
        'In plane': '105_IP_STRIPE_24JUNE26.DAT',
    },

    # 'Si': {
    #     'Out of plane': 'Si_substrate_MvsH.DAT',
    #     'In plane': 'Si_substrate_MvsH.DAT',
    # },

}

# --- Data directory ---
DATA_DIR = Path(r'C:\Users\emmab\Projects\Vector_Tomography\data_dir\VSM_data')

# --- Sample dimensions (same for both samples) ---
LENGTH_MM    = 3.8   # mm
WIDTH_MM     = 4.1   # mm

THICKNESS_NM_LIST  = [33, 66, 51, 45, 1000000]    # nm
THICKNESS_NM = dict(zip(FILES.keys(), THICKNESS_NM_LIST))

DW_WIDTH_NM_LIST  = [30, 23, 30, 25]    # nm
DW_WIDTH_NM = dict(zip(FILES.keys(), DW_WIDTH_NM_LIST))

# --- PPMS-specific settings ---
# Column names exactly as they appear in the header row after [Data]
H_COL_NAME = 'Magnetic Field (Oe)'
M_COL_NAME = 'Moment (emu)'
DATA_MARKER = '[Data]'          # line that signals the start of the data block
DELIMITER   = ','               # PPMS files are comma-separated

In [ ]:
# ============================================================
# Cell 3 — Volume calculation
# ============================================================

# Convert all dimensions to cm (CGS, consistent with emu)
length_cm    = LENGTH_MM    * 1e-1   # mm  → cm
width_cm     = WIDTH_MM     * 1e-1   # mm  → cm

VOLUME_CM3 = {}   # { sample_name: volume in cm³ }

print("── Sample volumes ──────────────────────────────────────")
print(f"  Lateral area: {LENGTH_MM} mm × {WIDTH_MM} mm")
print()

for sample_name, thickness_nm in THICKNESS_NM.items():
    thickness_cm              = thickness_nm * 1e-7   # nm → cm
    vol                       = length_cm * width_cm * thickness_cm
    VOLUME_CM3[sample_name]   = vol
    print(f"  {sample_name}:  {thickness_nm} nm  →  V = {vol:.4e} cm³")

print("────────────────────────────────────────────────────────")

In [ ]:
# ============================================================
# Cell 4 — Data loading helper
# ============================================================
def load_mvh(filepath, h_col_name=H_COL_NAME, m_col_name=M_COL_NAME,
             data_marker=DATA_MARKER, delimiter=DELIMITER):

    filepath = DATA_DIR / Path(filepath)   # ← only change: prepend DATA_DIR

    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath.resolve()}")

    with open(filepath, encoding='utf-8-sig') as f:
        lines = f.readlines()

    marker_idx = next(
        (i for i, line in enumerate(lines) if line.strip() == data_marker),
        None
    )
    if marker_idx is None:
        raise ValueError(f"'{data_marker}' not found in {filepath.name}. "
                         "Check DATA_MARKER in Cell 2.")

    col_header_line = lines[marker_idx + 1].strip().split(delimiter)
    col_header_line = [c.strip() for c in col_header_line]

    try:
        h_idx = col_header_line.index(h_col_name)
        m_idx = col_header_line.index(m_col_name)
    except ValueError as e:
        raise ValueError(
            f"{filepath.name}: column not found — {e}\n"
            f"Available columns: {col_header_line}"
        )

    rows = []
    for line in lines[marker_idx + 2:]:
        parts = line.strip().split(delimiter)
        try:
            rows.append((float(parts[h_idx]), float(parts[m_idx])))
        except (ValueError, IndexError):
            continue

    if not rows:
        raise ValueError(f"{filepath.name}: no numeric data rows found after header.")

    data = np.array(rows)
    return data[:, 0], data[:, 1]

In [ ]:
# ============================================================
# Cell 5 — Load data and normalise by volume
# ============================================================

dataset = {}

for sample_name, paths in FILES.items():
    dataset[sample_name] = {}
    vol = VOLUME_CM3[sample_name]          # ← each sample gets its own volume
    print(f"\n{sample_name}  (V = {vol:.4e} cm³)")
    print("─" * 50)

    for condition, filepath in paths.items():
        H, M_raw   = load_mvh(filepath)
        M_norm     = M_raw / vol           # emu → emu cm⁻³

        if sample_name == '[Co(2)/Ni(7)]': 
            H = H[40:]
            M_norm = M_norm[40:]

        else:
            H=H
            M_norm = M_norm

        
        dataset[sample_name][condition] = (H, M_norm)
        print(f"  [{condition:15s}]  {Path(filepath).name:45s}  "
              f"{len(H):>5d} points   M_max = {M_norm.max():.3e} emu/cm³")

In [ ]:
def is_in_plane(condition):
    key = condition.lower().replace("-", "").replace("_", "").replace(" ", "")
    return "inplane" in key

def subtract_diamagnetic(H, M, fit_frac=0.7, h_fit_min=None):
    """Fit M = chi*H + Ms*sign(H) on the high-field points (both branches, both signs)
    and subtract only the chi*H term. Returns (M_sub, chi, Ms)."""
    H, M = np.asarray(H, float), np.asarray(M, float)
    h_min = h_fit_min if h_fit_min is not None else fit_frac * np.nanmax(np.abs(H))
    mask = (np.abs(H) >= h_min) & np.isfinite(H) & np.isfinite(M)
    if mask.sum() < 4:
        raise ValueError(f"Only {mask.sum()} points above |H|={h_min:.3g}; lower FIT_FRAC.")

    # Separate saturation term for +H and -H so Ms doesn't bias the slope
    A = np.column_stack([H[mask], np.sign(H[mask])])
    (chi, Ms), *_ = np.linalg.lstsq(A, M[mask], rcond=None)
    return M - chi * H, chi, Ms



fracs = np.arange(0.4, 0.96, 0.05)

for sample_name, curves in dataset.items():
    fig, (ax_chi, ax_ms) = plt.subplots(1, 2, figsize=(10, 4))

    for condition, (H, M) in curves.items():
        chis, Mss, ok = [], [], []
        for f in fracs:
            try:
                _, chi, Ms = subtract_diamagnetic(H, M, f)
                chis.append(chi); Mss.append(Ms); ok.append(f)
            except ValueError:
                pass
        ax_chi.plot(ok, chis, 'o-', label=condition)
        ax_ms.plot(ok, Mss, 'o-', label=condition)

    ax_chi.set(xlabel='FIT_FRAC', ylabel='chi (emu/cm³/Oe)', title=f'{sample_name}: chi')
    ax_ms.set(xlabel='FIT_FRAC', ylabel='Ms (emu/cm³)', title='Ms')
    ax_chi.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------- User settings ----------------
OOP_KEY, IP_KEY = 'Out of plane', 'In plane'
OOP_PLATEAU_MIN_DEFAULT = 6000     # Oe; OOP is saturated and flat above this
OOP_PLATEAU_MIN = {                # per-sample override (check the fit plot!)
    # '[Co(2)/Ni(7)]': 6000,
}
IP_FIT_MIN = {                     # Oe; lower bound of the IP fit window, per sample
    '[Co(4)/Ni(5)/Pt(2)]': 16000,
}
IP_FIT_MIN_DEFAULT = 15000
# -----------------------------------------------

def fit_free(H, M, h_min):
    """M = chi*H + Ms*sign(H) on |H| >= h_min. Returns chi, Ms."""
    H, M = np.asarray(H, float), np.asarray(M, float)
    m = (np.abs(H) >= h_min) & np.isfinite(H) & np.isfinite(M)
    A = np.column_stack([H[m], np.sign(H[m])])
    (chi, Ms), *_ = np.linalg.lstsq(A, M[m], rcond=None)
    return chi, Ms

def fit_chi_fixed_Ms(H, M, h_min, Ms_ref):
    """Fit only chi with Ms fixed. Returns chi, rms residual on the window."""
    H, M = np.asarray(H, float), np.asarray(M, float)
    m = (np.abs(H) >= h_min) & np.isfinite(H) & np.isfinite(M)
    y = M[m] - Ms_ref * np.sign(H[m])
    chi = (H[m] @ y) / (H[m] @ H[m])
    return chi, np.sqrt(np.mean((y - chi * H[m])**2))

for sample_name, curves in dataset.items():
    if OOP_KEY not in curves or IP_KEY not in curves:
        continue
    H_o, M_o = (np.asarray(a, float) for a in curves[OOP_KEY])
    H_i, M_i = (np.asarray(a, float) for a in curves[IP_KEY])
    h_oop = OOP_PLATEAU_MIN.get(sample_name, OOP_PLATEAU_MIN_DEFAULT)
    h_min = IP_FIT_MIN.get(sample_name, IP_FIT_MIN_DEFAULT)

    # ── OOP: fit slope + Ms on the plateau, subtract the slope ──
    chi_oop, Ms_oop = fit_free(H_o, M_o, h_oop)
    M_o_corr = M_o - chi_oop * H_o
    curves[OOP_KEY + '_dia_sub'] = (H_o, M_o_corr)

    # ── IP: same as before, Ms fixed to the OOP value ──
    chi_free, Ms_free = fit_free(H_i, M_i, h_min)
    chi_fix, rms = fit_chi_fixed_Ms(H_i, M_i, h_min, Ms_oop)
    curves[IP_KEY + '_dia_sub'] = (H_i, M_i - chi_fix * H_i)

    print(f"{sample_name}: OOP chi = {chi_oop:.3e}, Ms_OOP = {Ms_oop:.1f} | "
          f"IP free fit: chi = {chi_free:.3e}, Ms = {Ms_free:.1f} "
          f"({100*(Ms_free/Ms_oop-1):+.1f}%) | "
          f"IP fixed-Ms fit: chi = {chi_fix:.3e}, rms = {rms:.1f}")



    fig, (ax1) = plt.subplots(1, 1, figsize=(6, 4))

    # Both corrected
    ax1.plot(H_o, M_o, color='#1f77b4', label='OOP raw')
    ax1.plot(H_i, curves[IP_KEY][1], color='#d62728', label='IP raw')
    ax1.axhline( Ms_oop, color='gray', ls='--', lw=0.9)
    ax1.axhline(-Ms_oop, color='gray', ls='--', lw=0.9)
    ax1.set(xlabel='H (Oe)', ylabel='M (emu cm$^{-3}$)', title=f'IP vs OOP \n {sample_name}')
    ax1.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    # ── Check plots ──
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

    # OOP: raw, fit, corrected
    Hp = np.linspace(-np.max(np.abs(H_o)), np.max(np.abs(H_o)), 400)
    ax1.plot(H_o, M_o, color='#1f77b4', label='OOP raw')
    ax1.plot(H_o, M_o_corr, color='navy', lw=1, label='OOP corrected')
    ax1.plot(Hp, chi_oop*Hp + Ms_oop*np.sign(Hp), 'k--', lw=1, label='plateau fit')
    for s in (-1, 1):
        ax1.axvline(s*h_oop, color='gray', ls=':')
    ax1.set(xlabel='H (Oe)', ylabel='M (emu cm$^{-3}$)', title=f'{sample_name}: OOP fit window')
    ax1.legend(fontsize=8)

    # IP: raw with fit window and both fitted lines
    Hp = np.linspace(-np.max(np.abs(H_i)), np.max(np.abs(H_i)), 400)
    ax2.plot(H_i, M_i, color='#d62728', label='IP raw')
    ax2.plot(Hp, chi_free*Hp + Ms_free*np.sign(Hp), 'k--', lw=1, label='free fit')
    ax2.plot(Hp, chi_fix*Hp + Ms_oop*np.sign(Hp), 'g-', lw=1, label='Ms fixed to OOP')
    for s in (-1, 1):
        ax2.axvline(s*h_min, color='gray', ls=':')
    ax2.set(xlabel='H (Oe)', title='IP fit window')
    ax2.legend(fontsize=8)

    # Both corrected
    ax3.plot(H_o, M_o_corr, color='#1f77b4', label='OOP corrected')
    ax3.plot(H_i, curves[IP_KEY + '_dia_sub'][1], color='#d62728', label='IP corrected')
    ax3.axhline( Ms_oop, color='gray', ls='--', lw=0.9)
    ax3.axhline(-Ms_oop, color='gray', ls='--', lw=0.9)
    ax3.set(xlabel='H (Oe)', title='corrected IP vs OOP')
    ax3.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    fig, (ax1) = plt.subplots(1, 1, figsize=(6, 4))

    # Both corrected
    ax1.plot(H_o, M_o_corr, color='#1f77b4', label='OOP corrected')
    ax1.plot(H_i, curves[IP_KEY + '_dia_sub'][1], color='#d62728', label='IP corrected')
    ax1.axhline( Ms_oop, color='gray', ls='--', lw=0.9)
    ax1.axhline(-Ms_oop, color='gray', ls='--', lw=0.9)
    ax1.set(xlabel='H (Oe)', ylabel='M (emu cm$^{-3}$)', title=f'IP vs OOP \n {sample_name}')
    ax1.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# One color per sample — extend this list if you add more samples
SAMPLE_COLORS = {
    '[Co(4)/Ni(5)/Pt(2)]': 'green',   
    '[Co(2)/Ni(7)]': 'mediumvioletred',   
    '[Co(6)/Ni(6)/Pt(5)]': 'gold',    
    '[Co(10)/Pt(5)]': 'blue',  
    
}

# Line style is fixed per condition (both are solid here, adjust if preferred)
CONDITION_LINESTYLE = {
    'Out of plane_dia_sub': '-',
    'In plane_dia_sub':     '-',
}

CONDITIONS = ['Out of plane_dia_sub', 'In plane_dia_sub']

for condition in CONDITIONS:

    fig, ax = plt.subplots(figsize=(8, 5))

    for sample_name, curves in dataset.items():
        if condition not in curves:
            print(f"  Warning: '{condition}' not found for {sample_name}, skipping.")
            continue

        H, M = curves[condition]
        ax.plot(H, M,
                color     = SAMPLE_COLORS[sample_name],
                linestyle = CONDITION_LINESTYLE[condition],
                linewidth = 1.8,
                label     = sample_name)

    # Reference lines
    ax.axhline(0, color='k', linewidth=0.7, linestyle=':')
    ax.axvline(0, color='k', linewidth=0.7, linestyle=':')

    ax.set_xlabel('H  (Oe)',            fontsize=13)
    ax.set_ylabel(r'M  (emu cm$^{-3}$)', fontsize=13)
    ax.set_title(f'{condition}', fontsize=13)
    ax.legend(fontsize=11, framealpha=0.9)
    ax.tick_params(labelsize=11)

    plt.tight_layout()

    # save_name = f"{condition.replace(' ', '_')}_all_samples_MvH.png"
    # plt.savefig(save_name, dpi=150, bbox_inches='tight')
    # print(f"Saved: {save_name}")

    plt.show()

# Ms vs t

In [ ]:
# ============================================================
# Cell A — Extraction helpers
# ============================================================

 
def _sort_by_H(H, M):
    """Sort arrays by field value."""
    idx = np.argsort(H)
    return H[idx], M[idx]
 
def _split_branches(H, M):
    """
    Split a hysteresis loop into the descending branch (H high→low)
    and ascending branch (H low→high).
    Returns (H_desc, M_desc), (H_asc, M_asc).
    """
    # Peak index — where the sweep reverses
    peak = np.argmax(H)
    H_desc, M_desc = H[:peak+1], M[:peak+1]
    H_asc,  M_asc  = H[peak:],   M[peak:]
    return (H_desc, M_desc), (H_asc, M_asc)
 
def _interp_zero_crossing(H, M):
    """
    Return the H value where M crosses zero (coercive field).
    Uses linear interpolation between the two adjacent points
    that straddle M=0. Returns np.nan if no crossing found.
    """
    sign_changes = np.where(np.diff(np.sign(M)))[0]
    if len(sign_changes) == 0:
        return np.nan
    i = sign_changes[0]
    # Linear interpolation
    h0, h1 = H[i], H[i+1]
    m0, m1 = M[i], M[i+1]
    if m1 == m0:
        return np.nan
    return h0 - m0 * (h1 - h0) / (m1 - m0)

def get_Mr(H, M):
    """Average |M| at H=0 over all zero-field crossings (i.e. avg of +Mr and -Mr)."""
    H, M = np.asarray(H, float), np.asarray(M, float)

    neg = H < 0
    i = np.where(neg[:-1] != neg[1:])[0]   # segments that straddle H=0

    # linear interpolation of M at H=0 on each segment
    M0 = M[i] - H[i] * (M[i+1] - M[i]) / (H[i+1] - H[i])

    return np.mean(np.abs(M0))

In [ ]:
def extract_oop_params(H, M, top_frac=0.05):
    """
    Extract Ms, Hc, Mr, Mr/Ms from an OOP (easy-axis) loop.
 
    Strategy
    --------
    Ms  : Mean of |M| in the top `top_frac` fraction of |H| values on
          both the positive and negative saturation sides.
          (Robust against the diamagnetic slope from the substrate —
          even small residuals average out at high field.)
 
    Hc  : Average of the two zero-crossings (one on each branch).
          Uses linear interpolation for sub-point precision.
 
    Mr  : Average |M| at H≈0 on each branch (interpolated at H=0).
          Because of the labyrinth domain nucleation, Mr << Ms.
 
    Returns a dict of floats (all in the units of H and M passed in).
    """
    H, M = np.array(H, dtype=float), np.array(M, dtype=float)
    (H_desc, M_desc), (H_asc, M_asc) = _split_branches(H, M)
 
    # --- Ms: saturated tails -----------------------------------------
    H_abs = np.abs(H)
    thresh = np.quantile(H_abs, 1 - top_frac)
    mask_sat = H_abs >= thresh
    Ms = np.mean(np.abs(M[mask_sat]))
 
    # --- Hc: zero crossings on each branch ---------------------------
    Hc_desc = abs(_interp_zero_crossing(H_desc, M_desc))
    Hc_asc  = abs(_interp_zero_crossing(H_asc,  M_asc))
    valid_Hc = [h for h in [Hc_desc, Hc_asc] if not np.isnan(h)]
    Hc = np.mean(valid_Hc) if valid_Hc else np.nan
 
    # --- Mr: M at H=0 on each branch ---------------------------------
#     # Descending branch: interpolate at H=0
#     def interp_at_zero(Hb, Mb):
#         # Find region around H=0 — sort first
#         idx = np.argsort(Hb)
#         Hb, Mb = Hb[idx], Mb[idx]
#         if Hb[0] > 0 or Hb[-1] < 0:
#             return np.nan
#         f = interp1d(Hb, Mb, kind='linear', bounds_error=False)
#         return float(f(0.0))


 
#     Mr_desc = abs(interp_at_zero(H_desc, M_desc))
#     Mr_asc  = abs(interp_at_zero(H_asc,  M_asc))
#     valid_Mr = [m for m in [Mr_desc, Mr_asc] if not np.isnan(m)]
#     Mr = np.mean(valid_Mr) if valid_Mr else np.nan
 
    Mr = get_Mr(H, M)
    MrMs = Mr / Ms if Ms > 0 else np.nan
    
#     MrMs = Mr / Ms if Ms > 0 else np.nan
 
    return dict(Ms=Ms, Hc=Hc,  Mr=Mr, MrMs=MrMs)

In [ ]:


trapz = getattr(np, "trapezoid", None) or np.trapz   

def area_H_vs_M(H, M, Hc, Ms, n=1000):
    """
    Integral of (H - Hc) dM from M=0 to Ms.

    Uses the ascending branch (dH > 0) directly, and the descending branch
    flipped through the origin (H,M -> -H,-M), so both branches contribute.
    Works regardless of where the sweep starts.
    """
    dH = np.diff(H)
    asc = dH > 0
    Hm = np.r_[H[:-1][asc], -H[:-1][~asc]]
    Mm = np.r_[M[:-1][asc], -M[:-1][~asc]]

    order = np.argsort(Mm)                      # H as a function of M
    M_grid = np.linspace(0, Ms, n)
    H_grid = np.interp(M_grid, Mm[order], Hm[order])

    return trapz(H_grid - Hc, M_grid)


def calc_Keff(H_ip, M_ip, H_oop, M_oop):
    H_ip,  M_ip  = np.asarray(H_ip, float),  np.asarray(M_ip, float)
    H_oop, M_oop = np.asarray(H_oop, float), np.asarray(M_oop, float)

    oop = extract_oop_params(H_oop, M_oop)      # your existing function
    ip  = extract_oop_params(H_ip,  M_ip)

    Ms = oop["Ms"]                               # same Ms used for both integrals
    A_ip  = area_H_vs_M(H_ip,  M_ip,  ip["Hc"],  Ms)
    A_oop = area_H_vs_M(H_oop, M_oop, oop["Hc"], Ms)

    return dict(Ms_oop=Ms, Ms_IP = ip['Ms'], Hc_IP=ip["Hc"], Hc_OOP=oop["Hc"],
                A_IP=A_ip, A_OOP=A_oop, Keff=(A_ip - A_oop) * 0.1)



In [ ]:
# ─────────────────────────────────────────────────────────────
#  IP parameter extraction
# ─────────────────────────────────────────────────────────────
def extract_ip_params(H, M, Ms_oop,
                      hk_window_frac=0.15,
                      sat_frac=0.08):
    """
    Extract Hk (anisotropy field), Ku, Keff from an IP (hard-axis) loop.
 
    Strategy
    --------
    The hard-axis loop has a linear regime at low fields and then
    bends over toward saturation. The anisotropy field Hk is the
    intercept of the initial linear slope with Ms (the OOP saturation
    value), i.e. the field at which the linear extrapolation reaches Ms.
 
    Because the Si substrate adds a diamagnetic (negative) linear
    background at high fields, we use the *low-field linear region*
    only (|H| < hk_window_frac * H_max) to fit the slope, then
    extrapolate to Ms_oop.
 
    Ku  = (1/2) * mu0 * Ms^2   — shape anisotropy (demagnetisation) alone
         ... but more usefully here:
    Ku  = (1/2) * Hk * Ms      (in CGS: erg/cm³, with H in Oe, M in emu/cm³)
    Keff = Ku - (1/2) * mu0 * Ms^2
         In CGS: Keff = Ku - 2π * Ms²   [erg/cm³]
 
    Note: Ms_oop must be in the same units as M (emu/cm³).
    """
    H, M = np.array(H, dtype=float), np.array(M, dtype=float)
    H, M = _sort_by_H(H, M)
 
    H_max = np.max(np.abs(H))
 
    # Use the average of both branches in the linear window
    win = H_max * hk_window_frac
    mask_lin = np.abs(H) <= win
    if mask_lin.sum() < 4:
        win = H_max * 0.20
        mask_lin = np.abs(H) <= win
 
    H_lin, M_lin = H[mask_lin], M[mask_lin]
    # Fit linear slope dM/dH in the low-field region
    coeffs = np.polyfit(H_lin, M_lin, 1)
    slope = coeffs[0]   # emu cm⁻³ Oe⁻¹
 
    # Hk: where the linear extrapolation = Ms_oop
    # slope * Hk + intercept = Ms  →  Hk = (Ms - intercept) / slope
    intercept = coeffs[1]
    if abs(slope) < 1e-20:
        Hk = np.nan
    else:
        Hk = (Ms_oop - intercept) / slope
        # Hk should be positive; if not, try negative branch
        if Hk < 0:
            Hk = (-Ms_oop - intercept) / slope
        Hk = abs(Hk)
 
    # CGS energy densities (erg/cm³)
    # Ku  = (1/2) * Hk * Ms   [Oe * emu/cm³ = erg/cm³]
    Ku   = 0.5 * Hk * Ms_oop
 
    # Keff = Ku - 2π Ms²   (shape anisotropy = 2π Ms² in CGS)
    Keff = Ku - 2.0 * np.pi * Ms_oop**2
 
    return dict(Hk=Hk, Ku=Ku, Keff=Keff, ip_slope=slope)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# MU0 = 4e-7 * np.pi          # T·m/A  (J/m³ per (A/m)²)
# EMU_CM3_TO_A_M = 1e3        # 1 emu/cm³ = 1000 A/m

# # ── add Q to params ──────────────────────────────────────────
# for name, p in params.items():
#     Ms_SI = p['Ms'] * EMU_CM3_TO_A_M            # A/m (same OOP Ms used in the Keff integrals)
#     E_shape = 0.5 * MU0 * Ms_SI**2              # J/m³  (shape anisotropy energy density)
#     p['Q'] = p['Keff'] / E_shape + 1            # p['Keff'] in J/m³

#     print(f"{name:25s}  Keff = {p['Keff']:.3e} J/m³   "
#           f"½μ0Ms² = {E_shape:.3e} J/m³   Q = {p['Q']:.3f}")

# # ── plot Q vs thickness ──────────────────────────────────────
# fig, ax = plt.subplots(figsize=(6, 4.5))

# names = sorted(params, key=lambda n: THICKNESS_NM[n])
# t = np.array([THICKNESS_NM[n] for n in names])
# Q = np.array([params[n]['Q'] for n in names])

# ax.plot(t, Q, color='gray', lw=1, zorder=1)     # guide to the eye
# for n, ti, qi in zip(names, t, Q):
#     ax.scatter(ti, qi, s=80, color=SAMPLE_COLORS[n], edgecolor='k', zorder=2, label=n)

# ax.axhline(1, color='k', ls='--', lw=0.9)       # Q = 1  <->  Keff = 0
# ax.set_xlabel('Thickness  (nm)', fontsize=12)
# ax.set_ylabel(r'Q = $K_{eff}/(\frac{1}{2}\mu_0 M_s^2) + 1$', fontsize=12)
# ax.legend(fontsize=8, framealpha=0.9)
# ax.tick_params(labelsize=11)
# plt.tight_layout()
# plt.show()

In [ ]:
params = {}  # { sample_name: { param: value } }

print("=" * 65)
print("  MAGNETIC PARAMETER EXTRACTION")
MU0 = 4e-7 * np.pi          # T·m/A  (J/m³ per (A/m)²)
EMU_CM3_TO_A_M = 1e3        # 1 emu/cm³ = 1000 A/m
print("=" * 65)


A_EX_PJ_LIST = [10.24, -1.85, 7.24, 6.22]  # in pJ/m
A_EX_PJ = dict(zip(dataset.keys(), A_EX_PJ_LIST))   # {sample_name: A in pJ/m}


for sample_name, curves in dataset.items():
    p = {}
    t_nm = THICKNESS_NM[sample_name]
    p['thickness_nm'] = t_nm
    print(f"\n  {sample_name}  ({t_nm} nm)")
    print("  " + "─" * 55)

    # ── OOP ──────────────────────────────────────────────────
    if 'Out of plane_dia_sub' in curves:
        H_oop, M_oop = curves['Out of plane_dia_sub']
        oop = extract_oop_params(H_oop, M_oop, top_frac=0.05)
        p.update(oop)
        print(f"    OOP:")
        print(f"      Ms   = {oop['Ms']:.3e}  emu/cm³")
        print(f"      Hc   = {oop['Hc']:.1f}  Oe")
        print(f"      Mr   = {oop['Mr']:.3e}  emu/cm³")
        print(f"      Mr/Ms= {oop['MrMs']:.3f}")
    else:
        print("    OOP data not found — skipping.")
        oop = {}

    # ── IP ───────────────────────────────────────────────────
    if 'In plane_dia_sub' in curves and 'Ms' in p:
        H_ip, M_ip = curves['In plane_dia_sub']
        H_oop, M_oop = curves['Out of plane_dia_sub']

        ip = calc_Keff(H_ip, M_ip, H_oop, M_oop)
        
        Ms_SI = oop['Ms'] * EMU_CM3_TO_A_M            # A/m (same OOP Ms used in the Keff integrals) using oop 
        Kd = 0.5 * MU0 * Ms_SI**2    # J/m³  (shape anisotropy energy density)
        ip['Q'] =   ip['Keff'] / Kd + 1       # p['Keff'] in J/m³
        ip['Kd'] = Kd
        Ku = ip['Keff'] + Kd
        ip['Ku'] = Ku

        p.update(ip)

        # ip = extract_ip_params(H_ip, M_ip, Ms_oop=p['Ms'],
        #                        hk_window_frac=0.15)
        # p.update(ip)
        # print(f"    IP:")
        print(f"      A_IP   = {ip['A_IP']:.1f}  ")
        print(f"      A_OOP   = {ip['A_OOP']:.1e} ")
        print(f"      Keff = {ip['Keff']:.3e}  erg/cm³")
        print(f"      Q = {ip['Q']:.3e}")
        print(f"      Kd = {ip['Kd']:.3e}")
        print(f"      Ku = {ip['Ku']:.3e}")
    else:
        print("    IP data or Ms not available — skipping.")

    # ── Exchange stiffness ───────────────────────────────────
    p['A_ex'] = A_EX_PJ[sample_name] * 1e-12          # pJ/m -> J/m
    print(f"      A_ex = {p['A_ex']:.2e}  J/m  ({A_EX_PJ[sample_name]} pJ/m)")

    params[sample_name] = p

In [ ]:
# ============================================================
# Cell D — Diagnostic overlay plots
#           (check the fits before trusting the numbers)
# ============================================================
 
for sample_name, curves in dataset.items():
    p   = params[sample_name]
    t   = THICKNESS_NM[sample_name]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{sample_name}  ({t} nm)", fontsize=13, fontweight='bold')
 
    # ── Left: OOP ────────────────────────────────────────────
    ax = axes[0]
    if 'Out of plane_dia_sub' in curves:
        H, M = curves['Out of plane_dia_sub']
        ax.plot(H, M, color=SAMPLE_COLORS[sample_name], lw=1.6, label='OOP data')
        # Mark Ms
        Ms = p.get('Ms', np.nan)
        ax.axhline( Ms, color='gray', lw=0.9, ls='--', label=f'Ms = {Ms:.2e}')
        ax.axhline(-Ms, color='gray', lw=0.9, ls='--')
        # Mark Hc
        Hc = p.get('Hc', np.nan)
        ax.axvline( Hc, color='tomato',   lw=1.0, ls=':', label=f'Hc = {Hc:.0f} Oe')
        ax.axvline(-Hc, color='tomato',   lw=1.0, ls=':')
        # Mark Mr
        Mr = p.get('Mr', np.nan)
        ax.axhline( Mr, color='steelblue', lw=0.9, ls=':', label=f'Mr = {Mr:.2e}')
        ax.axhline(-Mr, color='steelblue', lw=0.9, ls=':')

 
    ax.axhline(0, color='k', lw=0.6, ls=':')
    ax.axvline(0, color='k', lw=0.6, ls=':')
    ax.set_xlabel('H  (Oe)', fontsize=12)
    ax.set_ylabel(r'M  (emu cm$^{-3}$)', fontsize=12)
    ax.set_title('Out of plane', fontsize=12)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.tick_params(labelsize=11)
 
    # ── Right: IP ────────────────────────────────────────────
    ax = axes[1]
    if 'In plane' in curves:
        H, M = curves['In plane_dia_sub']
        ax.plot(H, M, color=SAMPLE_COLORS[sample_name], lw=1.6, label='IP data')
 
        # Draw the linear fit used for Hk
        # slope = p.get('ip_slope', np.nan)
        Ms_IP    = p.get('Ms_IP', np.nan)
        Hc_IP    = p.get('Hc_IP', np.nan)
  
        ax.axhline( Ms_IP, color='gray',    lw=0.9, ls='--', label=f'Ms_IP = {Ms_IP:.2e}')
        ax.axhline(-Ms_IP, color='gray',    lw=0.9, ls='--')
        ax.axvline( Hc_IP, color='darkorange', lw=1.0, ls=':', label=f'Hc_IP = {Hc_IP:.0f} Oe')
        ax.axvline(-Hc_IP, color='darkorange', lw=1.0, ls=':')
 
    ax.axhline(0, color='k', lw=0.6, ls=':')
    ax.axvline(0, color='k', lw=0.6, ls=':')
    ax.set_xlabel('H  (Oe)', fontsize=12)
    ax.set_ylabel(r'M  (emu cm$^{-3}$)', fontsize=12)
    ax.set_title('In plane', fontsize=12)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.tick_params(labelsize=11)
 
    plt.tight_layout()
    plt.show()
 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

trapz = getattr(np, "trapezoid", None) or np.trapz


def H_of_M(H, M, Ms, n=500):
    """
    H as a function of M on a grid from 0 to Ms (first quadrant).
    Ascending branch used directly, descending branch flipped through the origin.
    """
    H, M = np.asarray(H, float), np.asarray(M, float)
    asc = np.diff(H) > 0
    Hm = np.r_[H[:-1][asc], -H[:-1][~asc]]
    Mm = np.r_[M[:-1][asc], -M[:-1][~asc]]
    order = np.argsort(Mm)
    M_grid = np.linspace(0, Ms, n)
    return M_grid, np.interp(M_grid, Mm[order], Hm[order])


for sample_name, curves in dataset.items():
    p = params[sample_name]
    t = THICKNESS_NM[sample_name]
    col = SAMPLE_COLORS[sample_name]

    Ms    = p.get('Ms', np.nan)       # OOP Ms used as the upper limit for both integrals
    Hc    = p.get('Hc', np.nan)
    Hc_IP = p.get('Hc_IP', np.nan)

    has_oop = 'Out of plane_dia_sub' in curves
    has_ip  = 'In plane_dia_sub' in curves     # key you actually plot from

    # Pre-compute the integrand curves
    if has_oop:
        H_oop, M_oop = curves['Out of plane_dia_sub']
        Mg, Hg_oop = H_of_M(H_oop, M_oop, Ms)
        A_oop = trapz(Hg_oop - Hc, Mg)
    if has_ip:
        H_ip, M_ip = curves['In plane_dia_sub']
        Mg, Hg_ip = H_of_M(H_ip, M_ip, Ms)
        A_ip = trapz(Hg_ip - Hc_IP, Mg)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle(f"{sample_name}  ({t} nm)", fontsize=13, fontweight='bold')

    # ── Left: OOP ────────────────────────────────────────────
    ax = axes[0]
    if has_oop:
        ax.plot(H_oop, M_oop, color=col, lw=1.6, label='OOP data')
        Mr = p.get('Mr', np.nan)
        ax.axhline( Ms, color='gray', lw=0.9, ls='--', label=f'Ms = {Ms:.2e}')
        ax.axhline(-Ms, color='gray', lw=0.9, ls='--')
        ax.axvline( Hc, color='tomato', lw=1.0, ls=':', label=f'Hc = {Hc:.0f} Oe')
        ax.axvline(-Hc, color='tomato', lw=1.0, ls=':')
        ax.axhline( Mr, color='steelblue', lw=0.9, ls=':', label=f'Mr = {Mr:.2e}')
        ax.axhline(-Mr, color='steelblue', lw=0.9, ls=':')
        # Integral: area between H=Hc and the curve, M from 0 to Ms
        ax.fill_betweenx(Mg, Hc, Hg_oop, color=col, alpha=0.3,
                         label=f'∫(H−Hc)dM = {A_oop:.2e}')
    ax.axhline(0, color='k', lw=0.6, ls=':')
    ax.axvline(0, color='k', lw=0.6, ls=':')
    ax.set_xlabel('H  (Oe)', fontsize=12)
    ax.set_ylabel(r'M  (emu cm$^{-3}$)', fontsize=12)
    ax.set_title('Out of plane', fontsize=12)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.tick_params(labelsize=11)

    # ── Middle: IP ───────────────────────────────────────────
    ax = axes[1]
    if has_ip:
        ax.plot(H_ip, M_ip, color=col, lw=1.6, label='IP data (dia-sub)')
        ax.axhline( Ms, color='gray', lw=0.9, ls='--', label=f'Ms (OOP) = {Ms:.2e}')
        ax.axhline(-Ms, color='gray', lw=0.9, ls='--')
        ax.axvline( Hc_IP, color='darkorange', lw=1.0, ls=':', label=f'Hc_IP = {Hc_IP:.0f} Oe')
        ax.axvline(-Hc_IP, color='darkorange', lw=1.0, ls=':')
        ax.fill_betweenx(Mg, Hc_IP, Hg_ip, color=col, alpha=0.3,
                         label=f'∫(H−Hc)dM = {A_ip:.2e}')
    ax.axhline(0, color='k', lw=0.6, ls=':')
    ax.axvline(0, color='k', lw=0.6, ls=':')
    ax.set_xlabel('H  (Oe)', fontsize=12)
    ax.set_ylabel(r'M  (emu cm$^{-3}$)', fontsize=12)
    ax.set_title('In plane', fontsize=12)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.tick_params(labelsize=11)

    # ── Right: subtraction (first quadrant, Hc-shifted) ──────
    ax = axes[2]
    if has_oop and has_ip:
        x_ip, x_oop = Hg_ip - Hc_IP, Hg_oop - Hc
        ax.plot(x_ip,  Mg, color='darkorange', lw=1.8, label=f'IP:  A = {A_ip:.2e}')
        ax.plot(x_oop, Mg, color='steelblue',  lw=1.8, label=f'OOP: A = {A_oop:.2e}')
        # Region between the curves = A_IP − A_OOP = Keff
        ax.fill_betweenx(Mg, x_oop, x_ip, where=(x_ip >= x_oop),
                         color='seagreen', alpha=0.4, label='IP > OOP  (adds to Keff)')
        ax.fill_betweenx(Mg, x_oop, x_ip, where=(x_ip <  x_oop),
                         color='crimson', alpha=0.4, label='IP < OOP  (subtracts)')
        ax.set_title(f'Keff = A_IP − A_OOP = {A_ip - A_oop:.3e} erg/cm³', fontsize=12)
    ax.set_xlabel('H − Hc  (Oe)', fontsize=12)
    ax.set_ylabel(r'M  (emu cm$^{-3}$)', fontsize=12)
    ax.set_ylim(0, Ms * 1.05)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.tick_params(labelsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Cell E — Parameter vs thickness plots
# ============================================================
 
# Ordered by thickness for the x-axis
sample_order = sorted(params.keys(), key=lambda s: THICKNESS_NM[s])
 
x_vals   = [THICKNESS_NM[s] for s in sample_order] 
x_labels = [f"{THICKNESS_NM[s]} nm\n{s}" for s in sample_order]
x_pos    = list(range(len(sample_order))) 
colors   = [SAMPLE_COLORS[s] for s in sample_order]
 
def _param_plot(param_key, ylabel, title, scale=1.0, ylog=False):
    """Generic helper — one parameter panel."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    y_vals = [params[s].get(param_key, np.nan) * scale for s in sample_order]
    ax.scatter(x_pos, y_vals, c=colors, s=90, zorder=5, edgecolors='k', linewidths=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.tick_params(labelsize=10)
    if param_key == 'Q': 
        ax.axhline(1, color='k', lw=0.6, ls=':')
    else:
        ax.axhline(0, color='k', lw=0.6, ls=':')
    if ylog:
        ax.set_yscale('log')
    plt.tight_layout()
    plt.show()
 
# ── OOP parameters ───────────────────────────────────────────
_param_plot('Ms',   r'$M_s$  (emu cm$^{-3}$)',            r'Saturation magnetization $M_s$')
_param_plot('Hc',   r'$H_c$  (Oe)',                        r'Coercive field $H_c$')
_param_plot('Mr',   r'$M_r$  (emu cm$^{-3}$)',            r'Remanent magnetization $M_r$')
# _param_plot('MrMs', r'$M_r / M_s$',                        r'Squareness $M_r/M_s$')
 
# ── IP / anisotropy parameters ───────────────────────────────
# _param_plot('Hk',   r'$H_k$  (Oe)',                        r'Anisotropy field $H_k$')
# _param_plot('Ku',   r'$K_u$  (erg cm$^{-3}$)',            r'Uniaxial anisotropy $K_u$')
_param_plot('Keff', r'$K_\mathrm{eff}$  (J/m$^{-3}$)', r'Effective anisotropy $K_\mathrm{eff}$')
_param_plot('Kd', r'$K_\mathrm{d}$  (J/m$^{-3}$)', r'Shape anisotropy $K_\mathrm{u}$')
_param_plot('Ku', r'$K_\mathrm{u}$  (J/m$^{-3}$)', r'uniaxial anisotropy $K_\mathrm{u}$')

_param_plot('Q', r'$Q$', r'Quality Factor $Q$')

In [ ]:

# rows = []
# for name, p in params.items():
#     row = {'keff_sample': name, 'thickness_nm': THICKNESS_NM[name]}
#     # keep only scalar results (skips arrays, strings, etc.)
#     row.update({k: float(v) for k, v in p.items()
#                 if isinstance(v, (int, float, np.number))})
#     rows.append(row)

# keff_df = pd.DataFrame(rows).sort_values('thickness_nm')
# keff_df.to_csv('Magnetometry_params_NOV26.csv', index=False)
# print(keff_df)

In [ ]:
params['[Co(4)/Ni(5)/Pt(2)]'].keys()

In [ ]:

def D_thr(d_nm, Ms_emu, Ku, A):
    """
    Threshold DMI, Eq. (23):  D = 2 mu0 Ms^2 / ( pi^2/(d ln2) + pi*sqrt((Ku + mu0 Ms^2/2)/A) )

    d_nm   : film thickness (nm), scalar or array
    Ms_emu : saturation magnetization (emu/cm^3)
    Ku     : uniaxial anisotropy (J/m^3)
    A      : exchange stiffness (J/m)
    returns: D_thr in mJ/m^2
    """
    d  = np.asarray(d_nm, float) * 1e-9              # m
    Ms = Ms_emu * EMU_CM3_TO_A_M                     # A/m
    num = 2 * MU0 * Ms**2
    den = np.pi**2 / (d * np.log(2)) + np.pi * np.sqrt((Ku + 0.5 * MU0 * Ms**2) / A)
    return 1e3 * num / den                           # J/m^2 -> mJ/m^2


def plot_D_thr(params, d_range_nm=(1, 100), n=300, show_measured=True,
               figsize=(7, 5)):
    """
    Plot D_thr(d) for every sample in params, with a marker at its actual thickness.

    Per sample, reads from params[name]:
      'Ms' (emu/cm^3), 'thickness_nm', 'color'
      'A_ex' (J/m)
      'Ku' (J/m^3)   -> if missing, computed as Keff + 0.5*mu0*Ms^2
      'D' (mJ/m^2)   -> optional measured DMI, plotted as a star if show_measured
    """
    d = np.linspace(*d_range_nm, n)

    fig, ax = plt.subplots(figsize=figsize)
    for name, p in params.items():
        Ms = p.get('Ms', np.nan)
        if 'Ku' in p and np.isfinite(p['Ku']):
            Ku = p['Ku']
        else:
            Ku = p.get('Keff', np.nan) + 0.5 * MU0 * (Ms * EMU_CM3_TO_A_M)**2
        A = p.get('A_ex', np.nan)
        if not np.all(np.isfinite([Ms, Ku, A])):
            print(f"Skipping {name}: missing Ms, Ku/Keff, or A_ex")
            continue

        col = p.get('color', None)
        ax.plot(d, D_thr(d, Ms, Ku, A), color=col, lw=1.8)

        t = p['thickness_nm']
        Dt = D_thr(t, Ms, Ku, A)
        ax.scatter(t, Dt, s=100, color=col, edgecolors='k', zorder=5, label=name)
        print(f"{name:12s} d = {t:5.1f} nm   Ms = {Ms:7.1f} emu/cm³   "
              f"Ku = {Ku:.3e} J/m³   A = {A:.2e} J/m   D_thr = {float(Dt):.3f} mJ/m²")

        if show_measured and 'D' in p and np.isfinite(p['D']):
            ax.scatter(t, p['D'], s=160, marker='*', color=col, edgecolors='k',
                       zorder=6)

    ax.set_xlabel('Film thickness  d  (nm)', fontsize=16)
    ax.set_ylabel(r'$D_\mathrm{thr}$  (mJ m$^{-2}$)', fontsize=16)
    ax.tick_params(labelsize=13)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=10, framealpha=0.9)
    plt.tight_layout()
    plt.show()




# Usage
plot_D_thr(params)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MU0 = 4e-7 * np.pi          # T·m/A
EMU_CM3_TO_A_M = 1e3        # 1 emu/cm³ = 1000 A/m

# Exchange stiffness per sample, in J/m (e.g. 10 pJ/m = 10e-12). Used only if
# params[name] has no 'A_ex' key. Replace with your values.
A_EX = {
    # '270': 10e-12,
}
A_EX_DEFAULT = 10e-12


def D_thr(d_nm, Ms_emu, Ku, A):
    """
    Threshold DMI, Eq. (23):  D = 2 mu0 Ms^2 / ( pi^2/(d ln2) + pi*sqrt((Ku + mu0 Ms^2/2)/A) )

    d_nm   : film thickness (nm), scalar or array
    Ms_emu : saturation magnetization (emu/cm^3)
    Ku     : uniaxial anisotropy (J/m^3)
    A      : exchange stiffness (J/m)
    returns: D_thr in mJ/m^2
    """
    d  = np.asarray(d_nm, float) * 1e-9              # m
    Ms = Ms_emu * EMU_CM3_TO_A_M                     # A/m
    num = 2 * MU0 * Ms**2
    den = np.pi**2 / (d * np.log(2)) + np.pi * np.sqrt((Ku + 0.5 * MU0 * Ms**2) / A)
    return 1e3 * num / den                           # J/m^2 -> mJ/m^2


def plot_D_thr(params, d_range_nm=(1, 100), n=300, A_ex=None, show_measured=True,
               figsize=(7, 5)):
    """
    Plot D_thr(d) for every sample in params, with a marker at its actual thickness.

    Per sample, reads from params[name]:
      'Ms' (emu/cm^3), 'thickness_nm', 'color'
      'Ku' (J/m^3)        -> if missing, computed as Keff + 0.5*mu0*Ms^2
      'A_ex' (J/m)        -> if missing, taken from A_ex dict, then A_EX_DEFAULT
      'D' (mJ/m^2)        -> optional measured DMI, plotted as a star if show_measured
    """
    A_ex = A_ex if A_ex is not None else A_EX
    d = np.linspace(*d_range_nm, n)

    fig, ax = plt.subplots(figsize=figsize)
    for name, p in params.items():
        Ms = p.get('Ms', np.nan)
        if 'Ku' in p and np.isfinite(p['Ku']):
            Ku = p['Ku']
        else:
            Ku = p.get('Keff', np.nan) + 0.5 * MU0 * (Ms * EMU_CM3_TO_A_M)**2
        A = p.get('A_ex', A_ex.get(name, A_EX_DEFAULT))
        if not np.all(np.isfinite([Ms, Ku, A])):
            print(f"Skipping {name}: missing Ms, Ku/Keff, or A")
            continue

        col = p.get('color', None)
        ax.plot(d, D_thr(d, Ms, Ku, A), color=col, lw=1.8)

        t = p['thickness_nm']
        Dt = D_thr(t, Ms, Ku, A)
        ax.scatter(t, Dt, s=100, color=col, edgecolors='k', zorder=5, label=name)
        print(f"{name:12s} d = {t:5.1f} nm   Ms = {Ms:7.1f} emu/cm³   "
              f"Ku = {Ku:.3e} J/m³   A = {A:.2e} J/m   D_thr = {float(Dt):.3f} mJ/m²")

        if show_measured and 'D' in p and np.isfinite(p['D']):
            ax.scatter(t, p['D'], s=160, marker='*', color=col, edgecolors='k',
                       zorder=6)

    ax.set_xlabel('Film thickness  d  (nm)', fontsize=16)
    ax.set_ylabel(r'$D_\mathrm{thr}$  (mJ m$^{-2}$)', fontsize=16)
    ax.tick_params(labelsize=13)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=10, framealpha=0.9)
    plt.tight_layout()
    plt.show()


# Usage
plot_D_thr(params)

In [ ]:
# ── 1. Compute D_thr at each sample's own thickness and store in params ──
def add_D_thr(params):
    for name, p in params.items():
        Ms = p.get('Ms', np.nan)
        Ku = p.get('Ku', p.get('Keff', np.nan) + 0.5 * MU0 * (Ms * EMU_CM3_TO_A_M)**2)
        A  = p.get('A_ex', np.nan)
        t  = p.get('thickness_nm', THICKNESS_NM.get(name, np.nan))
        p['D_thr'] = float(D_thr(t, Ms, Ku, A)) if np.all(np.isfinite([Ms, Ku, A, t])) else np.nan

add_D_thr(params)

# ── 2. Same helper as before, with an optional overlay (e.g. measured D) ──
sample_order = sorted(params.keys(), key=lambda s: THICKNESS_NM[s])
x_labels = [f"{THICKNESS_NM[s]} nm\n{s}" for s in sample_order]
x_pos    = list(range(len(sample_order)))
colors   = [SAMPLE_COLORS[s] for s in sample_order]

def _param_plot(param_key, ylabel, title, scale=1.0, ylog=False, overlay_key=None):
    """One parameter panel. overlay_key: second params key drawn as stars (e.g. 'D')."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    y_vals = [params[s].get(param_key, np.nan) * scale for s in sample_order]
    ax.scatter(x_pos, y_vals, c=colors, s=90, zorder=5, edgecolors='k', linewidths=0.6,
               label=param_key)
    if overlay_key:
        y2 = [params[s].get(overlay_key, np.nan) * scale for s in sample_order]
        ax.scatter(x_pos, y2, c=colors, s=170, marker='*', zorder=6, edgecolors='k',
                   linewidths=0.6, label=overlay_key)
        ax.legend(fontsize=10)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.tick_params(labelsize=10)
    ax.axhline(1 if param_key == 'Q' else 0, color='k', lw=0.6, ls=':')
    if ylog:
        ax.set_yscale('log')
    plt.tight_layout()
    plt.show()

# ── 3. Plot ──
_param_plot('D_thr', r'$D_\mathrm{thr}$  (mJ m$^{-2}$)', r'Threshold DMI $D_\mathrm{thr}$')

# If you have measured DMI in params[name]['D'] (mJ/m^2), compare directly:
# _param_plot('D_thr', r'$D$  (mJ m$^{-2}$)', r'$D_\mathrm{thr}$ vs measured $D$', overlay_key='D')

In [ ]:
# rows = []
# for name, p in params.items():
#     row = {'keff_sample': name, 'thickness_nm': THICKNESS_NM[name]}
#     # keep only scalar results (skips arrays, strings, etc.)
#     row.update({k: float(v) for k, v in p.items()
#                 if isinstance(v, (int, float, np.number))})
#     rows.append(row)

# keff_df = pd.DataFrame(rows).sort_values('thickness_nm')
# keff_df.to_csv('Magnetometry_params_NOV26.csv', index=False)
# print(keff_df)

In [ ]:
stop

# MFM

In [ ]:
# ============================================================
# Cell 7 — Additional imports for MFM analysis
# ============================================================
from PIL import Image
from scipy.ndimage import gaussian_filter

try:
    import tifffile
    HAS_TIFFFILE = True
except ImportError:
    HAS_TIFFFILE = False
    print("⚠  tifffile not installed — falling back to PIL.\n"
          "   For full scientific TIFF support: pip install tifffile")

In [ ]:
# ============================================================
# Cell 8 — MFM configuration  (edit this cell)
# ============================================================

MFM_FILES = {
    'Sample 452': {
        'pre':  '452_pre_annealed.tiff',
        'post': '452_annealed.tiff',
    },
    'Sample 270': {
        'pre':  '270_pre_annealed.tiff',
        'post': '270_annealed.tiff',
    },
}
GAUSSIAN_SIGMA   = 0.01    # smoothing strength in pixels; increase for more blur
SCAN_SIZE_UM     = 5.0   # physical scan size in µm — set to None for pixel units
APPLY_WINDOW     = True   # Hann window before FFT to suppress edge ringing
NORMALIZE_RADIAL = True   # normalise each radial profile to its own max for shape comparison
SKIP_DC_PX       = 3      # pixels to exclude near DC in radial profile (suppresses mean offset)
RADIAL_LOG_SCALE = False  # set True for log y-axis on radial profile plot

MFM_CMAP  = 'cividis'        # MFM image colourmap — 'RdBu' or 'bwr' emphasise phase contrast
FFT_CMAP  = 'inferno'     # FFT colourmap

In [ ]:
# ============================================================
# Cell 9 (corrected) — Helper functions
# ============================================================

def load_tiff(filepath):
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath.resolve()}")
    if HAS_TIFFFILE:
        img = tifffile.imread(str(filepath)).astype(np.float32)
    else:
        img = np.array(Image.open(filepath)).astype(np.float32)
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    elif img.ndim != 2:
        raise ValueError(f"Unexpected image shape: {img.shape}")
    return img


def normalise_image(img, plow=1, phigh=99):
    vmin, vmax = np.percentile(img, [plow, phigh])
    return np.clip((img - vmin) / (vmax - vmin + 1e-12), 0, 1)


def compute_fft(image, apply_window=APPLY_WINDOW):
    ny, nx = image.shape
    img    = image.copy()
    if apply_window:
        win = np.outer(np.hanning(ny), np.hanning(nx))
        img = img * win
    fft_shift = np.fft.fftshift(np.fft.fft2(img))
    fft_mag   = np.abs(fft_shift)
    fft_log   = np.log1p(fft_mag)
    return fft_log, fft_mag


def radial_profile(fft_mag, scan_size_um=SCAN_SIZE_UM, skip_dc_px=SKIP_DC_PX):
    ny, nx = fft_mag.shape
    cy, cx = ny // 2, nx // 2

    y_idx, x_idx = np.indices((ny, nx))
    r_px = np.sqrt((x_idx - cx)**2 + (y_idx - cy)**2).astype(int)

    r_flat = r_px.ravel()
    m_flat = fft_mag.ravel()

    profile = np.bincount(r_flat, weights=m_flat) / np.bincount(r_flat)

    if scan_size_um is not None:
        r_axis    = np.arange(len(profile)) / scan_size_um
        nyquist   = 0.5 * max(nx, ny) / scan_size_um
        freq_unit = r'$\mu$m$^{-1}$'          # ← pure LaTeX, no unicode µ
        xy_unit   = r'$\mu$m'
    else:
        r_axis    = np.arange(len(profile)) / max(nx, ny)
        nyquist   = 0.5
        freq_unit = r'cycles px$^{-1}$'
        xy_unit   = 'px'

    mask = (np.arange(len(profile)) >= skip_dc_px) & (r_axis <= nyquist)
    return r_axis[mask], profile[mask], freq_unit, xy_unit

In [ ]:
# ============================================================
# Cell 10 (corrected) — Load images, smooth, FFT, radial profiles
# ============================================================

mfm_raw     = {}
mfm_smooth  = {}
mfm_fft_log = {}
mfm_fft_mag = {}
mfm_radial  = {}

print("Loading and processing MFM images...")
print("─" * 60)

for sample_name, paths in MFM_FILES.items():
    for store in [mfm_raw, mfm_smooth, mfm_fft_log, mfm_fft_mag, mfm_radial]:
        store[sample_name] = {}

    for condition, filepath in paths.items():
        raw              = load_tiff(filepath)
        smooth           = gaussian_filter(raw, sigma=GAUSSIAN_SIGMA)
        f_log, f_mag     = compute_fft(smooth)
        r_ax, prof, freq_unit, xy_unit = radial_profile(f_mag)   # 4 return values now

        mfm_raw[sample_name][condition]     = raw
        mfm_smooth[sample_name][condition]  = smooth
        mfm_fft_log[sample_name][condition] = f_log
        mfm_fft_mag[sample_name][condition] = f_mag
        mfm_radial[sample_name][condition]  = (r_ax, prof, freq_unit, xy_unit)

        print(f"  {sample_name} [{condition:4s}]  "
              f"{Path(filepath).name:35s}  shape: {raw.shape}")

print("─" * 60)
print(f"  Gaussian sigma = {GAUSSIAN_SIGMA} px | "
      f"Hann window: {APPLY_WINDOW} | "
      f"Scan size: {SCAN_SIZE_UM} um")

In [ ]:
# ============================================================
# Cell 11 (corrected + complete) — MFM images + 2D FFTs
#
#   Row 0:  Pre-anneal image  |  Post-anneal image
#   Row 1:  Pre-anneal FFT    |  Post-anneal FFT
# ============================================================

LABEL = {'pre': 'Pre-anneal', 'post': 'Post-anneal'}

for sample_name in MFM_FILES:
    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    fig.suptitle(f'{sample_name}, MFM Images & 2D FFT',
                 fontsize=14, fontweight='bold', y=1.01)

    # Pull units from any stored radial profile entry (same for all conditions)
    _, _, freq_unit, xy_unit = next(iter(mfm_radial[sample_name].values()))

    for col, condition in enumerate(['pre', 'post']):
        smooth  = mfm_smooth[sample_name][condition]
        fft_log = mfm_fft_log[sample_name][condition]
        ny, nx  = smooth.shape

        # ── Row 0: MFM image ─────────────────────────────────────────
        ax = axes[0, col]

        if SCAN_SIZE_UM is not None:
            extent_img = [0, SCAN_SIZE_UM, SCAN_SIZE_UM, 0]
        else:
            extent_img = [0, nx, ny, 0]

        im_mfm = ax.imshow(normalise_image(smooth),
                           cmap=MFM_CMAP, extent=extent_img, aspect='equal')
        ax.set_title(LABEL[condition], fontsize=12, fontweight='bold')
        ax.set_xlabel(xy_unit, fontsize=11)
        ax.set_ylabel(xy_unit, fontsize=11)
        plt.colorbar(im_mfm, ax=ax, fraction=0.046, pad=0.04,
                     label='MFM phase (a.u.)')

        # ── Row 1: 2D FFT ────────────────────────────────────────────
        ax = axes[1, col]

        if SCAN_SIZE_UM is not None:
            fmax_x     = 0.5 * nx / SCAN_SIZE_UM
            fmax_y     = 0.5 * ny / SCAN_SIZE_UM
            extent_fft = [-fmax_x, fmax_x, -fmax_y, fmax_y]
        else:
            extent_fft = [-0.5, 0.5, -0.5, 0.5]

        im_fft = ax.imshow(fft_log, cmap=FFT_CMAP,
                           extent=extent_fft, origin='lower', aspect='equal')
        ax.set_title(f'2D FFT, {LABEL[condition]}', fontsize=12, fontweight='bold')
        ax.set_xlabel(freq_unit, fontsize=11)
        ax.set_ylabel(freq_unit, fontsize=11)
        plt.colorbar(im_fft, ax=ax, fraction=0.046, pad=0.04,
                     label=r'log(1 + |FFT|)')

    plt.tight_layout()
    # save_name = f"{sample_name.replace(' ', '_')}_MFM_FFT.png"
    # plt.savefig(save_name, dpi=150, bbox_inches='tight')
    # print(f"Saved: {save_name}")
    plt.show()

In [ ]:
# ============================================================
# Cell 12 — Radial FFT line cuts (one panel per sample,
#            pre- vs post-anneal overlaid)
# ============================================================

RADIAL_STYLE = {
    'pre':  dict(color='#1f77b4', linestyle='-',  linewidth=2.0, label='Pre-anneal'),
    'post': dict(color='#d62728', linestyle='--', linewidth=2.0, label='Post-anneal'),
}

fig, axes = plt.subplots(1, len(MFM_FILES), figsize=(7 * len(MFM_FILES), 5),
                         sharey=False)

# Make axes always iterable even if there is only one sample
if len(MFM_FILES) == 1:
    axes = [axes]

for ax, sample_name in zip(axes, MFM_FILES):

    for condition in ['pre', 'post']:
        r_ax, prof, freq_unit, _ = mfm_radial[sample_name][condition]

        # Optional: normalise each profile to its own max for shape comparison
        if NORMALIZE_RADIAL:
            prof = prof / prof.max()

        ax.plot(r_ax, prof, **RADIAL_STYLE[condition])

    ax.set_xlabel(freq_unit, fontsize=12)
    y_label = 'Norm. |FFT| (a.u.)' if NORMALIZE_RADIAL else '|FFT| (a.u.)'
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_title(f'{sample_name}, Radial FFT Profile', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, framealpha=0.9)
    ax.tick_params(labelsize=11)

    if RADIAL_LOG_SCALE:
        ax.set_yscale('log')

    ax.axhline(0, color='k', linewidth=0.6, linestyle=':')
    ax.set_xlim(left=0)

plt.tight_layout()
# save_name = 'all_samples_radial_FFT.png'
# plt.savefig(save_name, dpi=150, bbox_inches='tight')
# print(f"Saved: {save_name}")
plt.show()